# 10a | Machine Learning: Oral Review

The ML part of a DS / ML intern interview is mostly **conceptual**: explain how a model works, compare two models, say what you would do in a given situation. You are rarely asked to derive formulas. The goal is to answer each question clearly in 3–5 sentences and state the tradeoffs.

About 55 minutes.

> **How to use this lesson:** Answer each question **out loud** first, then open the reference answer to compare. Each answer ends with a short version you can say in an interview.

## 1. Generalization: training, validation, and test sets

- **Training set:** fit the model parameters;
- **Validation set (or cross-validation):** choose hyperparameters and models;
- **Test set:** use once at the very end to estimate generalization performance.

If you keep adjusting the model based on test-set results, the test set is no longer "unseen data."

**🗣 Say it out loud:** Explain the bias–variance tradeoff.

<details>
<summary><b>Reference answer</b></summary>

A model's expected prediction error decomposes into three parts: bias² + variance + irreducible noise. **Bias** comes from a model that is too simple to capture the true pattern (underfitting): both training and test error are high. **Variance** comes from a model that is too flexible and also learns the noise in the training data (overfitting): training error is very low, test error is high, and the fit changes a lot with a different training sample. Increasing model complexity lowers bias and raises variance. The goal is the complexity that minimizes their sum, chosen with a validation set or cross-validation.

**Short version:** Expected error decomposes into squared bias, variance, and irreducible noise. Simple models underfit with high bias; flexible models overfit with high variance. We tune complexity, usually through cross-validation, to minimize the sum.

</details>

**🗣 Say it out loud:** How do you tell that a model is overfitting? What can you do about it?

<details>
<summary><b>Reference answer</b></summary>

**Diagnosis:** Performance on the training set is far better than on the validation set; on the learning curve the gap between them is large and does not shrink as data grows.

**Remedies:** ① more data; ② lower model complexity (fewer features, shallower trees, fewer parameters); ③ regularization (L1 / L2, dropout, early stopping); ④ ensembles (bagging reduces variance); ⑤ feature selection or dimensionality reduction; ⑥ data augmentation; ⑦ check for data leakage (leakage inflates the validation score too, and you only find out in production).

Conversely, for underfitting: add features or interaction terms, use a more flexible model, weaken regularization.

**Short version:** A large gap between training and validation performance signals overfitting. Remedies: more data, a simpler model, regularization, early stopping, bagging, or feature selection. For underfitting, add features or use a more flexible model.

</details>

**🗣 Say it out loud:** How does K-fold cross-validation work? Why use it? What should you watch out for?

<details>
<summary><b>Reference answer</b></summary>

Split the training data into K parts; in turn, use one part for validation and the other K−1 for training. This gives K validation scores, which you average. It is more stable than a single split and uses the data more fully. K is usually 5 or 10.

**Watch out for:** ① all preprocessing (scaling, imputation, feature selection, target encoding) must be fit on the training part of each fold — put it in a Pipeline — otherwise it leaks; ② use stratified K-fold for imbalanced classes; ③ never shuffle time series randomly; use forward-rolling time-based splits; ④ when one user has multiple rows, split by user (GroupKFold), otherwise the same person appears in both training and validation; ⑤ after choosing hyperparameters with CV, still report final performance on an independent test set.

**Short version:** Split the training data into K folds, train on K minus one and validate on the held-out fold, rotating, then average. It gives a more stable estimate than a single split. All preprocessing must be fit inside each fold, and the split must respect time ordering and grouping.

</details>

**🗣 Say it out loud:** What is data leakage? Give two examples.

<details>
<summary><b>Reference answer</b></summary>

Training uses information that would **not be available** at the moment of a real prediction, so offline evaluation is inflated and the model fails in production.

Examples: ① **Target leakage:** a feature is a consequence of the outcome — e.g., predicting whether a user will pay using "amount spent in the next 30 days," or predicting a disease using "was the corresponding drug prescribed." ② **Preprocessing leakage:** scaling, imputation, or feature selection done on all data before splitting. ③ **Temporal leakage:** randomly splitting a time series, so the future is used to predict the past. ④ **Duplicate / group leakage:** records of the same user appear in both training and test sets.

Warning signs: one feature has an absurdly high importance, or the model looks too good to be true. A good habit: for every feature ask, "At the moment of prediction, does this value already exist?"

**Short version:** Leakage is when training uses information that wouldn't be available at prediction time, inflating offline metrics. Examples: a feature that's a consequence of the target, fitting a scaler on all data before splitting, random splits of time-ordered data, or the same user in both train and test.

</details>

## 2. Decision trees and ensembles

**Decision tree:** at each step pick one feature and one split point that make the two sides as "pure" as possible (Gini impurity or entropy for classification, variance / MSE for regression), and recurse.

- Pros: handles nonlinearity and interactions automatically; no scaling needed; easy to interpret;
- Cons: a single tree has very high variance — a small change in the data gives a completely different tree; prone to overfitting.

Controlling complexity: maximum depth, minimum samples per leaf, pruning.

**🗣 Say it out loud:** How does a random forest work? Why is it better than a single tree?

<details>
<summary><b>Reference answer</b></summary>

Random forest = **bagging + random feature subsets**. Each tree is trained on a bootstrap sample (sampling with replacement), and at each split only a random subset of features is considered for the best split. Finally, the trees' predictions are averaged (regression) or voted (classification).

Why it is better: a single deep tree has low bias but high variance. Averaging many trees reduces variance, and random feature selection **decorrelates** the trees, which makes averaging more effective (the more correlated the trees, the less averaging reduces variance).

Side benefits: samples not drawn for a tree (out-of-bag) can estimate generalization error; training parallelizes; it is insensitive to hyperparameters, so it makes a great baseline.

**Short version:** A random forest trains many deep trees on bootstrap samples, considering a random subset of features at each split, and averages them. Averaging reduces variance, and the feature subsampling decorrelates the trees so the averaging is more effective.

</details>

**🗣 Say it out loud:** What is the core difference between a random forest and gradient boosting?

<details>
<summary><b>Reference answer</b></summary>

**How they are built:** random forest trees are trained **independently, in parallel**, each one deep, then averaged; gradient boosting trees are built **sequentially** — each new tree fits the residuals of the current model (more generally, the negative gradient of the loss), correcting earlier errors step by step, and each tree is usually shallow.

**What they reduce:** random forests mainly reduce **variance**; boosting mainly reduces **bias** (while controlling variance with the learning rate, subsampling, etc.).

**In practice:** boosting (XGBoost, LightGBM) is usually more accurate on tabular data but has more hyperparameters (number of trees, learning rate, depth), overfits more easily, and needs early stopping; random forests are more robust and easier to tune.

**Short version:** Random forests build deep trees independently and average them, mainly reducing variance. Boosting builds shallow trees sequentially, each fitting the residual errors of the ensemble so far, mainly reducing bias. Boosting is usually more accurate on tabular data but needs more careful tuning.

</details>

**🗣 Say it out loud:** What are the important hyperparameters of a gradient boosting model (e.g., XGBoost / LightGBM)?

<details>
<summary><b>Reference answer</b></summary>

① **Number of trees** and **learning rate**: they work together — the smaller the learning rate, the more trees you need; usually set a small learning rate plus early stopping to choose the number of trees automatically; ② **tree depth / number of leaves**: controls the complexity of each tree and the order of interactions it can capture; ③ **minimum samples per leaf**: prevents splitting on noise; ④ **row and column subsampling ratios**: add randomness and reduce variance; ⑤ **L1 / L2 regularization terms**. A typical tuning order: fix a fairly large learning rate to get a rough number of trees, tune the tree structure, then the subsampling, and finally lower the learning rate.

**Short version:** The number of trees and the learning rate, which trade off against each other and are usually set with early stopping; tree depth or number of leaves; minimum samples per leaf; row and column subsampling; and the regularization terms.

</details>

## 3. Other supervised models

**🗣 Say it out loud:** How does kNN work? When does it perform poorly?

<details>
<summary><b>Reference answer</b></summary>

To predict, find the k training points closest to the new point; take a majority vote for classification or the average for regression. There is no explicit training step. Small k means high variance (overfitting); large k means high bias.

**When it performs poorly:** ① features on different scales (you must standardize first); ② high-dimensional data: in high dimensions all pairwise distances are similar, so "nearest neighbor" loses its meaning (curse of dimensionality); ③ many irrelevant features; ④ large datasets make prediction slow (distances to all training points); ⑤ class imbalance.

**Short version:** It predicts from the k closest training points by majority vote or average. It struggles with unscaled features, with high-dimensional data where distances become uninformative, with many irrelevant features, and it's slow at prediction time on large datasets.

</details>

**🗣 Say it out loud:** What does an SVM optimize? What is the kernel trick?

<details>
<summary><b>Reference answer</b></summary>

A linear SVM looks for the separating hyperplane that **maximizes the margin** between the two classes; only the points closest to the boundary (the support vectors) determine the result. A soft margin allows some points to cross the margin, with the parameter C controlling the penalty for violations: large C tries harder to classify training points correctly (narrow margin, prone to overfitting).

**Kernel trick:** implicitly map the data into a high-dimensional space and separate it linearly there, which corresponds to a nonlinear boundary in the original space. Because the algorithm only needs inner products between samples, a kernel function (e.g., RBF) computes the inner product in the high-dimensional space directly, without performing the mapping.

SVMs are used less on tabular data today (slow to train on large data, and tree models are usually better), but the concepts are still asked often.

**Short version:** An SVM finds the separating hyperplane with the largest margin, determined by the support vectors; C controls the penalty for margin violations. The kernel trick computes inner products in a high-dimensional feature space implicitly, giving nonlinear boundaries without explicit mapping.

</details>

**🗣 Say it out loud:** Which models need standardized features, and which don't?

<details>
<summary><b>Reference answer</b></summary>

**Need it:** distance-based models (kNN, k-means, SVM), gradient-descent-based models (neural networks, regularized linear / logistic regression), and PCA. **Don't need it:** tree models (decision trees, random forests, gradient boosting), because they only use the ordering of feature values and monotone transformations don't change the splits. For ordinary unregularized linear regression, standardization doesn't change the predictions, only the scale of the coefficients.

</details>

**🗣 Say it out loud:** How do you handle categorical features and missing values?

<details>
<summary><b>Reference answer</b></summary>

**Categorical features:** use one-hot encoding when there are few categories; with many categories (high cardinality, e.g., city or product ID) use target encoding (computed within folds to avoid leakage), frequency encoding, hashing, or a model that handles categories natively (LightGBM, CatBoost); ordinal categories can use integer encoding.

**Missing values:** first understand why they are missing. Impute numeric features with the median and **add a "was missing" indicator column**; treat missing as its own category for categorical features; model-based imputation is also an option. Gradient boosting libraries handle missing values natively. Compute imputation statistics from the training set only.

**Short version:** For categoricals: one-hot for low cardinality; target or frequency encoding, or native categorical support, for high cardinality. For missing values: understand why they're missing, impute with a median plus a missing indicator, and fit imputation on training data only.

</details>

## 4. Unsupervised learning

**🗣 Say it out loud:** What does PCA do? Why standardize before using it? What are its limitations?

<details>
<summary><b>Reference answer</b></summary>

PCA finds a set of mutually orthogonal directions (principal components) along which the projected data has **maximum variance**; this is equivalent to an eigendecomposition of the covariance matrix (or an SVD of the centered data). Keeping the first k components reduces dimensionality while retaining as much variance as possible.

**Standardization:** PCA is variance-based, and variables with large units naturally have large variance and would dominate the components. Unless the variables' units are already comparable, standardize first.

**Limitations:** ① captures only linear structure; ② unsupervised — it ignores y, and the direction of maximum variance is not necessarily the most useful for prediction; ③ components are linear combinations of the original variables and hard to interpret; ④ sensitive to outliers.

**Short version:** PCA finds orthogonal directions of maximum variance, equivalent to an eigendecomposition of the covariance matrix. Features should be standardized so large-unit variables don't dominate. It's linear, ignores the target, and the components can be hard to interpret.

</details>

**🗣 Say it out loud:** How does k-means work? What are its assumptions and drawbacks? How do you choose k?

<details>
<summary><b>Reference answer</b></summary>

**Algorithm:** randomly pick k centers → assign each point to the nearest center → update each center to the mean of its points → repeat until nothing changes. It minimizes the within-cluster sum of squared distances; no step increases this objective, so it always converges, but only to a local optimum.

**Assumptions / drawbacks:** implicitly assumes spherical clusters of similar size; sensitive to initialization (use k-means++ initialization and multiple runs); sensitive to feature scale and outliers; k must be specified in advance.

**Choosing k:** the elbow method (the bend in the within-cluster sum of squares as a function of k), silhouette scores, or business needs (e.g., the operations team can only run different strategies for 4 user groups).

**Short version:** Assign each point to its nearest centroid, recompute centroids as cluster means, and repeat until stable. It minimizes within-cluster squared distance, converges to a local optimum, assumes roughly spherical clusters, and is sensitive to scale, outliers, and initialization. Choose k with the elbow method, silhouette scores, or business constraints.

</details>

## 5. Optimization: gradient descent

**🗣 Say it out loud:** Explain gradient descent and the difference between its batch, stochastic, and mini-batch variants.

<details>
<summary><b>Reference answer</b></summary>

Gradient descent repeatedly updates the parameters in the **opposite** direction of the loss gradient: $\theta \leftarrow \theta - \eta\,\nabla L(\theta)$, where η is the learning rate. Too large a learning rate oscillates or even diverges; too small converges very slowly.

**Batch:** compute the gradient on all data each step — accurate direction, but each step is slow. **Stochastic (SGD):** one sample per step — fast updates, but noisy. **Mini-batch:** a small batch per step (e.g., 32–512), balancing the two; it is the standard in deep learning, and the noise also helps escape poor local optima. In practice, optimizers with momentum or adaptive learning rates (Adam) are common.

**Short version:** Gradient descent repeatedly moves parameters against the gradient of the loss, scaled by a learning rate. Batch uses all data per step, SGD uses one example, and mini-batch uses small batches, which is the practical default. Optimizers like Adam add momentum and adaptive step sizes.

</details>

## 6. Deep learning: the bare minimum

DS intern interviews usually don't go deep on deep learning; ML intern interviews do. At minimum, be able to answer each of the following in a sentence or two.

**🗣 Say it out loud:** How is a neural network trained? What do activation functions, dropout, and batch normalization each do?

<details>
<summary><b>Reference answer</b></summary>

**Training:** a forward pass computes predictions and the loss; **backpropagation** uses the chain rule to compute the gradient of the loss with respect to every parameter; a gradient-descent-type optimizer updates the parameters.

**Activation functions (e.g., ReLU):** introduce nonlinearity. Without them, stacked linear layers are still one linear transformation. ReLU is cheap to compute and mitigates the vanishing-gradient problem of sigmoid / tanh.

**Dropout:** during training, randomly set the outputs of some neurons to 0, preventing neurons from co-adapting; it is a form of regularization and is not used at prediction time.

**Batch normalization:** standardize each layer's inputs over the mini-batch, which stabilizes training, allows larger learning rates, and has a mild regularizing effect.

Other ways to prevent overfitting: weight decay (L2), early stopping, data augmentation.

**Short version:** A forward pass computes the loss, backpropagation applies the chain rule to get gradients, and an optimizer updates the weights. Activations like ReLU add nonlinearity; dropout randomly zeroes units as regularization; batch norm normalizes layer inputs to stabilize and speed up training.

</details>

**🗣 Say it out loud:** In a sentence or two each: what data are CNNs, RNNs / LSTMs, and Transformers suited for, and what is the core idea of each?

<details>
<summary><b>Reference answer</b></summary>

**CNN:** images and other data with spatial structure. Convolution kernels that slide locally and share parameters extract local features, which are combined layer by layer into more abstract features.

**RNN / LSTM:** sequence data. Processes time steps one by one, carrying earlier information in a hidden state; LSTMs use gating to mitigate vanishing gradients over long sequences.

**Transformer:** sequence data; now the foundation of NLP and large language models. **Self-attention** lets every position in the sequence attend directly to every other position, with no sequential processing, so training parallelizes and long-range dependencies are captured better; positional encodings preserve order information.

**Short version:** CNNs use shared local filters for spatial data like images. RNNs and LSTMs process sequences step by step with a hidden state. Transformers use self-attention so every position attends to every other directly, which parallelizes well and captures long-range dependencies.

</details>

## 7. Model selection

**🗣 Say it out loud:** You have 100,000 rows of tabular data, 30 features, and a binary target, and the business wants a reasonably interpretable model. How do you start?

<details>
<summary><b>Reference answer</b></summary>

First pin down the goal and the evaluation metric (driven by business costs), check data quality, missingness, class balance, and possible leakage, and split appropriately (by time if there is a time dimension).

**Baseline:** regularized logistic regression with basic preprocessing. It is fast and interpretable, and gives a reference point for later models.

**Then:** gradient-boosted trees (LightGBM / XGBoost) to see how much nonlinearity and interactions add. If the gain is small, use logistic regression; if it is substantial, use the boosted trees and explain them with tools like SHAP and partial dependence plots.

The final choice depends on the tradeoff among accuracy, interpretability, calibration, latency, and maintenance cost — not a fixed "XGBoost is best."

**Short version:** I'd start by fixing the metric and split, then fit a regularized logistic regression as an interpretable baseline. Next I'd try gradient-boosted trees to see how much nonlinearity adds. If the gain is small I'd keep the simple model; if large, use the trees with SHAP for explanation. The choice balances accuracy, interpretability, calibration, and latency.

</details>

## Checkpoint

You can compare: linear / logistic regression, decision trees, random forests, gradient boosting, kNN, SVM, PCA, k-means. You can explain: bias–variance, remedies for overfitting, cross-validation, data leakage, gradient descent.

Move on to **10b** to do the whole workflow in scikit-learn.